# DAT NT-CLSM Pipeline

Run `00_project_config.ipynb` first. This notebook calls work scripts with the generated config file.

In [ ]:
from pathlib import Path
import os

# 工作脚本只从配置文件读取变量
project_dir = Path('/home/zhenzong2/analysis/neurotransmitter')
config_path = project_dir / 'config' / 'dat_config.yaml'
assert config_path.exists(), 'Run notebooks/00_project_config.ipynb first.'
os.chdir(project_dir)
print(config_path)

## Reference data

Run atlas, DAT maps, and LQT public resources first.

In [ ]:
!python scripts/fetch_reference_data.py --config {config_path} --maps
!python scripts/fetch_lqt_data.py --config {config_path}

## Prepare inputs

In [ ]:
!python scripts/prepare_inputs.py --config {config_path}

## NiiStat analyses

In [ ]:
!bash scripts/run_niistat_node_wm.sh --config {config_path}
!python scripts/postprocess_niistat.py --config {config_path}

## LQT-R edge analyses

In [ ]:
!Rscript scripts/install_lqt_r_deps.R --project-dir {project_dir}
!Rscript scripts/run_lqt_edges.R --config {config_path}

## DAT impact score

This step follows the SDC/FDC-style discovery-validation logic and writes patient-level lesion/DAT impact scores, 10-fold out-of-sample predictions, and bootstrap pairwise model comparisons.

In [ ]:
!python scripts/compute_dat_impact_scores.py --config {config_path}

## Collect results

In [ ]:
!python scripts/collect_results.py --config {config_path}